# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² Kilifi Mental Health survey dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is defined by a Croissant schema and contains multiple record sets and fields relevant to mental health indicators, demographics, and survey responses for residents of Kilifi County, Kenya.

### Dataset Source
The dataset source is provided as a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (by `@id`), their fields, and the fields' `@id` values.

In [ ]:
# List record sets and their fields using @id
print("Record sets in this dataset:")
record_set_list = list(metadata.record_sets)
for i, rs in enumerate(record_set_list, 1):
    print(f"{i}. RecordSet @id: {rs.id}  | name: {getattr(rs,'name', None)}")

    # List Fields
    print("   Fields and their @id values:")
    for field in rs.fields:
        print(f"     - {field.id}, name: {getattr(field,'name', None)}, type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load all records from each record set into a DataFrame for further analysis.
All references use record set and field `@id` values identified in the overview.

In [ ]:
# Extract all record sets data into DataFrames

# Get the list of record set @id values
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print("Loaded DataFrames for the following record sets:")
for rsid in dataframes:
    print(f"- {rsid} --> shape: {dataframes[rsid].shape}")

# Select the main survey record set for demonstration
# (We assume the main survey table is the first record set)
main_record_set_id = record_sets_ids[0]
print(f"\nColumns in {main_record_set_id} DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

# Preview data
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing fields, and grouping.
All fields are referenced by their `@id`.

In [ ]:
main_df = dataframes[main_record_set_id]

# Find a numeric field, e.g., the GAD-7 (Generalized Anxiety Disorder) score field id
# We search the fields for an integer/float field and pick one
gad7_field_id = None
group_field_id = None

for field in metadata.record_sets[0].fields:
    dtype = (getattr(field, 'data_type', '') or '').lower()
    name = str(getattr(field, 'name', '')).lower()
    if gad7_field_id is None and (dtype in ('integer', 'float', 'number') or 'score' in name or 'gad' in name):
        if (
            'gad' in name or 'anxiety' in name or 'phq' in name or 'score' in name or dtype in ('integer', 'float', 'number')
        ) and field.id in main_df.columns:
            gad7_field_id = field.id
    if group_field_id is None:
        # Try to find a demographic field for grouping (e.g., gender or education or age)
        if any(x in name for x in ['gender', 'sex', 'education', 'village', 'marital']):
            if field.id in main_df.columns:
                group_field_id = field.id

# If not found, pick any numeric
if not gad7_field_id:
    for col in main_df.select_dtypes(include=['float', 'int']).columns:
        gad7_field_id = col
        break
# If not found, skip EDA step

if gad7_field_id:
    print(f"Using numeric field '@id': {gad7_field_id}")
    field_vals = pd.to_numeric(main_df[gad7_field_id], errors='coerce')
    threshold = 10
    filtered_df = main_df[field_vals > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold} (N={len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the field
    filtered_df = filtered_df.copy()
    filtered_vals = pd.to_numeric(filtered_df[gad7_field_id], errors='coerce')
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_vals - filtered_vals.mean()) / filtered_vals.std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by selected demographic field if available
    if group_field_id and group_field_id in filtered_df.columns:
        print(f"\nGrouping by '{group_field_id}' (categorical field).")
        grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        print("Mean of", gad7_field_id, "by", group_field_id, ":")
        print(grouped_df.head())
else:
    print("No appropriate numeric field identified for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and its relationship with a demographic (if available). All field references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id and gad7_field_id in main_df.columns:
    # Plot the distribution of the selected field
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(main_df[gad7_field_id], errors='coerce').dropna(), kde=True)
    plt.title(f"Distribution of {gad7_field_id}")
    plt.xlabel(gad7_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(
            data=main_df,
            x=group_field_id,
            y=gad7_field_id
        )
        plt.title(f"{gad7_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(gad7_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² Kilifi Mental Health Survey dataset from its Croissant schema, reviewed its structure by `@id`, extracted its contents with `mlcroissant`, and performed basic exploratory data analysis and visualization. Further analysis (e.g., statistical modeling) can proceed with these loaded DataFrames referencing all fields by their Croissant `@id`s for reproducibility and schema adherence.